# Cultural Change From Texts Using Word Embeddings

This notebook is designed for Google Colab and classroom use. It demonstrates the core idea behind diachronic embedding studies: words get meaning from their relations to other words, and those relations can be compared across time slices.

The default demo uses a small synthetic corpus so everyone can run it quickly. At the end there is an optional live-data section using Library of Congress / Chronicling America newspaper OCR pages. Do not train on a full newspaper archive in class; that is a research-scale job.

## Two Student Tracks

**Track 1: Simple machines / standard Colab CPU.** Use the synthetic corpus and, optionally, a small Chronicling America sample. This teaches the method with low computational cost.

**Track 2: Better machines / Colab Pro / lab server.** Scale up the number of newspaper pages or use article-level datasets such as American Stories. For classical word2vec, CPU/RAM/disk matter more than GPU. GPU becomes useful if students use transformer models such as BERT or Sentence-BERT to embed contexts around target words.

Big GPUs are not required to understand diachronic word embeddings. They are useful for larger corpora or contextual embedding extensions.

## Computational Burden

This notebook is intentionally light. The synthetic demo should run in under a minute on Colab CPU. The optional newspaper demo downloads a small number of OCR pages and should usually run in a few minutes.

A serious project using millions of newspaper articles should be run on a server or in carefully prepared batches. For teaching, the goal is to understand the method, not to reproduce a full-scale paper during class.

In [ ]:
!pip -q install gensim pandas numpy matplotlib scikit-learn requests tqdm

import re
import time
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

random.seed(42)
np.random.seed(42)

## 1. Helper Functions

These functions tokenize text, train a small word2vec model, and compute a semantic dimension such as `modern - traditional` or `threat - friendly`.

In [ ]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return [w for w in text.split() if len(w) > 2]


def train_w2v(texts, vector_size=80, window=6, min_count=1, epochs=80):
    sentences = [tokenize(t) for t in texts]
    sentences = [s for s in sentences if len(s) >= 3]
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=2,
        sg=1,
        negative=10,
        epochs=epochs,
        seed=42,
    )
    return model


def in_vocab(model, words):
    return [w for w in words if w in model.wv]


def semantic_axis(model, positive_words, negative_words):
    pos = in_vocab(model, positive_words)
    neg = in_vocab(model, negative_words)
    if not pos or not neg:
        raise ValueError(f"Need words from both poles in vocabulary. Found positive={pos}, negative={neg}")
    pos_vec = np.mean([model.wv[w] for w in pos], axis=0)
    neg_vec = np.mean([model.wv[w] for w in neg], axis=0)
    axis = pos_vec - neg_vec
    return axis / np.linalg.norm(axis)


def projection_score(model, word, axis):
    if word not in model.wv:
        return np.nan
    v = model.wv[word]
    return float(np.dot(v / np.linalg.norm(v), axis))


def score_words(model, words, axis, label="score"):
    rows = []
    for w in words:
        rows.append({"word": w, label: projection_score(model, w, axis), "in_vocab": w in model.wv})
    return pd.DataFrame(rows)


def show_neighbors(model, word, topn=10):
    if word not in model.wv:
        print(f"'{word}' is not in vocabulary")
        return
    for neighbor, score in model.wv.most_similar(word, topn=topn):
        print(f"{neighbor:15s} {score:.3f}")

## 2. A Small Synthetic Example

The corpus below is artificial. That is useful pedagogically because we know what changed. In the early period, `immigrant` appears near danger and poverty language. In the later period, it appears near contribution and community language. We then test whether embeddings recover that change.

In [ ]:
early_templates = [
    "the immigrant was described as foreign poor dangerous disorderly suspicious",
    "newspapers linked immigrant neighborhoods with crime poverty threat and disorder",
    "the foreign worker was called cheap strange unamerican and risky",
    "reports warned that immigrant groups brought danger disease poverty and unrest",
    "the nation debated whether immigrant families were loyal or threatening",
    "city officials described immigrant districts as crowded poor and dangerous",
    "the article connected foreign arrivals with labor unrest and disorder",
]

late_templates = [
    "the immigrant was described as hardworking creative ambitious and loyal",
    "newspapers linked immigrant communities with family enterprise culture and contribution",
    "the foreign worker was called skilled productive modern and valuable",
    "reports praised immigrant groups for innovation service work and community life",
    "the nation debated immigrant rights citizenship belonging and opportunity",
    "city officials described immigrant districts as diverse vibrant and entrepreneurial",
    "the article connected foreign arrivals with renewal growth and cultural energy",
]

stable_templates = [
    "scientists studied medicine education technology and public health",
    "teachers discussed schools books learning and children",
    "farmers sold wheat corn milk and vegetables in the market",
    "artists created music paintings theater and literature",
]

# Anchor sentences keep the measurement vocabulary available in both periods.
# Without these anchors, one model may not know words from one pole of the axis.
anchor_templates = [
    "hardworking creative ambitious loyal valuable skilled vibrant are positive descriptive words",
    "dangerous poor disorderly suspicious threat risky unrest are negative descriptive words",
    "newspapers can describe groups with positive words or negative words",
]

early_texts = (early_templates * 80) + (stable_templates * 60) + (anchor_templates * 80)
late_texts = (late_templates * 80) + (stable_templates * 60) + (anchor_templates * 80)

early_model = train_w2v(early_texts)
late_model = train_w2v(late_texts)

print("Early period nearest neighbors for immigrant:")
show_neighbors(early_model, "immigrant")
print("\nLate period nearest neighbors for immigrant:")
show_neighbors(late_model, "immigrant")

## 3. Build a Cultural Dimension

Following the logic of semantic dimensions, define an axis from one set of words to another. Here we build a simplified `positive - negative` dimension.

In [ ]:
positive = ["hardworking", "creative", "ambitious", "loyal", "valuable", "skilled", "vibrant"]
negative = ["dangerous", "poor", "disorderly", "suspicious", "threat", "risky", "unrest"]

early_axis = semantic_axis(early_model, positive, negative)
late_axis = semantic_axis(late_model, positive, negative)

targets = ["immigrant", "foreign", "worker", "scientists", "teachers", "farmers", "artists"]

early_scores = score_words(early_model, targets, early_axis, "early_positive_score")
late_scores = score_words(late_model, targets, late_axis, "late_positive_score")

scores = early_scores.merge(late_scores, on=["word"], suffixes=("_early", "_late"))
scores[["word", "early_positive_score", "late_positive_score"]]

In [ ]:
plot_df = scores.dropna(subset=["early_positive_score", "late_positive_score"]).copy()
x = np.arange(len(plot_df))
width = 0.38

plt.figure(figsize=(10, 4))
plt.bar(x - width/2, plot_df["early_positive_score"], width, label="Early")
plt.bar(x + width/2, plot_df["late_positive_score"], width, label="Late")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(x, plot_df["word"], rotation=30, ha="right")
plt.ylabel("Projection on positive - negative axis")
plt.title("Synthetic Example: Association Change Across Time")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Visualize the Embedding Space

This PCA plot compresses the high-dimensional embedding space into two dimensions. Do not overinterpret the exact position; use it as an intuition-building visualization.

In [ ]:
words_to_plot = [
    "immigrant", "foreign", "worker", "dangerous", "poor", "threat", "unrest",
    "hardworking", "creative", "loyal", "valuable", "community", "contribution",
    "scientists", "teachers", "farmers", "artists"
]

def pca_plot(model, words, title):
    words = [w for w in words if w in model.wv]
    X = np.array([model.wv[w] for w in words])
    coords = PCA(n_components=2, random_state=42).fit_transform(X)
    plt.figure(figsize=(8, 6))
    plt.scatter(coords[:, 0], coords[:, 1])
    for w, (x, y) in zip(words, coords):
        plt.text(x, y, w, fontsize=10)
    plt.title(title)
    plt.axhline(0, color="lightgray", linewidth=0.8)
    plt.axvline(0, color="lightgray", linewidth=0.8)
    plt.tight_layout()
    plt.show()

pca_plot(early_model, words_to_plot, "Early Period Embeddings")
pca_plot(late_model, words_to_plot, "Late Period Embeddings")

## 5. Optional: Small Live Newspaper Demo

This section downloads a small number of OCR pages from Chronicling America. It is not a full research corpus. It is a quick demonstration of how real historical newspaper text can be collected and used.

Set `RUN_LIVE_DEMO = True` if you want to try it. If the API is slow or unavailable during class, skip this section.

In [ ]:
RUN_LIVE_DEMO = False

def search_chronicling_america(term, date1, date2, rows=20):
    url = "https://chroniclingamerica.loc.gov/search/pages/results/"
    params = {
        "andtext": term,
        "date1": date1,
        "date2": date2,
        "rows": rows,
        "format": "json",
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("items", [])


def fetch_ocr_text(item):
    # Chronicling America page IDs usually support /ocr.txt
    page_id = item.get("id") or item.get("url")
    if not page_id:
        return ""
    if not page_id.endswith("/"):
        page_id += "/"
    txt_url = page_id + "ocr.txt"
    r = requests.get(txt_url, timeout=30)
    if r.status_code != 200:
        return ""
    return r.text


def collect_small_newspaper_corpus(terms, date1, date2, rows_per_term=10, pause=0.2):
    records = []
    for term in tqdm(terms):
        try:
            items = search_chronicling_america(term, date1, date2, rows=rows_per_term)
        except Exception as e:
            print("Search failed", term, e)
            continue
        for item in items:
            try:
                text = fetch_ocr_text(item)
            except Exception:
                text = ""
            if len(text) > 500:
                records.append({
                    "query": term,
                    "date": item.get("date"),
                    "title": item.get("title"),
                    "url": item.get("url"),
                    "text": text,
                })
            time.sleep(pause)
    return pd.DataFrame(records)


if RUN_LIVE_DEMO:
    terms = ["china", "japan", "mexico", "germany", "turkey"]
    early_news = collect_small_newspaper_corpus(terms, 1900, 1910, rows_per_term=12)
    later_news = collect_small_newspaper_corpus(terms, 1945, 1955, rows_per_term=12)
    print(early_news.shape, later_news.shape)
    display(early_news.head())

In [ ]:
if RUN_LIVE_DEMO:
    # For real work, use much more data and stronger preprocessing.
    early_news_model = train_w2v(early_news["text"].tolist(), min_count=2, epochs=30)
    later_news_model = train_w2v(later_news["text"].tolist(), min_count=2, epochs=30)

    print("Early neighbors for china")
    show_neighbors(early_news_model, "china")
    print("\nLater neighbors for china")
    show_neighbors(later_news_model, "china")

In [ ]:
if RUN_LIVE_DEMO:
    threat_words = ["war", "danger", "threat", "enemy", "attack", "conflict"]
    friendly_words = ["peace", "friend", "ally", "trade", "cooperation", "agreement"]

    early_threat_axis = semantic_axis(early_news_model, threat_words, friendly_words)
    later_threat_axis = semantic_axis(later_news_model, threat_words, friendly_words)

    countries = ["china", "japan", "mexico", "germany", "turkey"]
    early_country_scores = score_words(early_news_model, countries, early_threat_axis, "early_threat_score")
    later_country_scores = score_words(later_news_model, countries, later_threat_axis, "later_threat_score")

    country_scores = early_country_scores.merge(later_country_scores, on="word")
    display(country_scores[["word", "early_threat_score", "later_threat_score"]])

## 6. Advanced Optional Track: Larger Newspaper Corpus

This section is for students with more time, RAM, disk space, and a stable internet connection. It uses the American Stories dataset, which provides article-level text extracted from Chronicling America.

Start with a few years and a few thousand articles. Do not load all years in a class session. A full diachronic project should be prepared as batch jobs and saved to disk. For classical word2vec, GPU is usually not the bottleneck; data download, preprocessing, CPU, RAM, and storage are.

In [ ]:
RUN_ADVANCED_AMERICAN_STORIES = False

if RUN_ADVANCED_AMERICAN_STORIES:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "pyarrow"])
    from datasets import load_dataset

    def load_american_stories_sample(years, max_articles_per_year=1000):
        """Load a small article-level sample from selected American Stories years."""
        ds = load_dataset(
            "dell-research-harvard/AmericanStories",
            "subset_years",
            year_list=[str(y) for y in years],
            trust_remote_code=True,
        )
        rows = []
        for year in years:
            split = ds[str(year)]
            n = min(max_articles_per_year, len(split))
            sample = split.shuffle(seed=42).select(range(n))
            for row in sample:
                article = row.get("article") or ""
                if len(article) > 300:
                    rows.append({
                        "year": int(year),
                        "date": row.get("date"),
                        "newspaper_name": row.get("newspaper_name"),
                        "headline": row.get("headline"),
                        "article": article,
                    })
        return pd.DataFrame(rows)

    # Increase these gradually if the machine can handle it.
    early_years = [1900, 1901]
    late_years = [1950, 1951]
    max_articles_per_year = 1000

    early_big = load_american_stories_sample(early_years, max_articles_per_year)
    late_big = load_american_stories_sample(late_years, max_articles_per_year)

    print("Early sample:", early_big.shape)
    print("Late sample:", late_big.shape)
    display(early_big.head())

In [ ]:
if RUN_ADVANCED_AMERICAN_STORIES:
    # min_count should be higher for larger real corpora.
    early_big_model = train_w2v(early_big["article"].tolist(), vector_size=100, window=8, min_count=10, epochs=10)
    late_big_model = train_w2v(late_big["article"].tolist(), vector_size=100, window=8, min_count=10, epochs=10)

    countries = ["china", "japan", "mexico", "germany", "turkey", "russia", "iran"]
    threat_words = ["war", "danger", "threat", "enemy", "attack", "conflict"]
    friendly_words = ["peace", "friend", "ally", "trade", "cooperation", "agreement"]

    early_big_axis = semantic_axis(early_big_model, threat_words, friendly_words)
    late_big_axis = semantic_axis(late_big_model, threat_words, friendly_words)

    early_country_scores = score_words(early_big_model, countries, early_big_axis, "early_threat_score")
    late_country_scores = score_words(late_big_model, countries, late_big_axis, "late_threat_score")
    country_scores = early_country_scores.merge(late_country_scores, on="word")
    display(country_scores[["word", "early_threat_score", "late_threat_score", "in_vocab_early", "in_vocab_late"]])

## Discussion Questions

1. What exactly does the model measure: culture, language, media discourse, or something else?
2. How would results change if we used local newspapers rather than national newspapers?
3. Why might OCR errors matter for historical embeddings?
4. What are the dangers of using binary dimensions such as `civilized - backward` or `modern - traditional`?
5. How could we validate the embedding results using surveys, hand coding, or historical knowledge?